In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from heston.pinn.heston_pinn_nd import HestonMultiAssetPINN
from utility.model import ModelConfig, EarlyStopping

In [2]:
n_assets = 2
K = 1.0    # Strike price
T = 1.0      # Time to maturity
r = 0.05     # Risk-free rate
kappa = 2.0  # Mean reversion rate
theta = 0.04 # Long-term variance
sigma_bar = 0.3  # Volatility of variance

sigmas = np.array([0.2, 0.25])  # Volatility of each asset

# Correlation matrix
rho_asset = 0 # Correlation between assets
corr = np.full((n_assets, n_assets), rho_asset)
np.fill_diagonal(corr, 1.0)

rho_cross = [0.3, 0.4]  # Correlation between stock price and variance for each asset

S_min = np.full(n_assets, 0.0)
V_min = 0.01

S0 = 1.0
v0 = 0.04

S_max = np.full(n_assets, 3 * S0)
V_max = 5 * v0


In [3]:
input_size = 4
hidden_sizes = [64, 64, 64, 64]
output_size = 1
activation = nn.Sigmoid()
learning_rate = 0.001

# Scheduler
step_size = 2000
gamma = 0.7

model_config = ModelConfig(
    input_size=input_size,
    hidden_sizes=hidden_sizes,
    output_size=output_size,
    activation=activation,
    learning_rate=learning_rate,
    step_size=step_size,
    gamma=gamma
)

loss_weights = {
    'variational': 5,
    'terminal': 5,
    'Smin': 3,
    'Smax': 3,
    'Vmin': 3,
    'Vmax': 3
}

In [ ]:
seed = 43
pinn = HestonMultiAssetPINN(model_config, seed)
pinn.set_params(K, r, T, kappa, theta, sigma_bar, sigmas, corr, rho_cross, S_min, S_max, V_min, V_max)
pinn.set_loss_weights(loss_weights)

early_stopping = EarlyStopping(patience=500, min_delta=1e-7)
pinn.train(batch_size=4096, epochs=30000, early_stopping=early_stopping)

Iteration 0 | Training Loss: 1.3230984210968018 | Validation Loss: 1.1577279567718506
Iteration 500 | Training Loss: 0.027801793068647385 | Validation Loss: 0.027689622715115547
Iteration 1000 | Training Loss: 0.0039567239582538605 | Validation Loss: 0.003976179286837578
Iteration 1500 | Training Loss: 0.002292326418682933 | Validation Loss: 0.0023275157436728477
Iteration 2000 | Training Loss: 0.0017421027878299356 | Validation Loss: 0.0018308210419490933
Iteration 2500 | Training Loss: 0.0013268725015223026 | Validation Loss: 0.0013415628345683217
Iteration 3000 | Training Loss: 0.0005946574383415282 | Validation Loss: 0.0005861171521246433
Iteration 3500 | Training Loss: 0.0001857435709098354 | Validation Loss: 0.0001951680169440806
Iteration 4000 | Training Loss: 9.695711196400225e-05 | Validation Loss: 9.925273479893804e-05
Iteration 4500 | Training Loss: 7.055807509459555e-05 | Validation Loss: 7.491148426197469e-05
Iteration 5000 | Training Loss: 5.44889408047311e-05 | Validatio

In [5]:
pinn.save('../models/heston_pinn_nd/42.pth')